## Importing Packages

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek

import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Load teh Data

In [3]:
data = load_breast_cancer()

X = data.data
y = data.target

In [4]:
print("Shape:", X.shape)
print("Target:", data.target_names)
print("Class Dst:", np.unique(y, return_counts=True))

Shape: (569, 30)
Target: ['malignant' 'benign']
Class Dst: (array([0, 1]), array([212, 357]))


## Train Ready

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

## Diff Exp Models

In [6]:
log_reg = LogisticRegression(
    C=1,
    solver="liblinear",
    random_state=42
)

log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.88      0.92        64
           1       0.93      0.98      0.95       107

    accuracy                           0.94       171
   macro avg       0.95      0.93      0.94       171
weighted avg       0.94      0.94      0.94       171



In [7]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.91      0.92        64
           1       0.94      0.96      0.95       107

    accuracy                           0.94       171
   macro avg       0.94      0.93      0.94       171
weighted avg       0.94      0.94      0.94       171



In [8]:
xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.92      0.95        64
           1       0.95      0.99      0.97       107

    accuracy                           0.96       171
   macro avg       0.97      0.96      0.96       171
weighted avg       0.97      0.96      0.96       171



In [9]:
smote = SMOTETomek(random_state=42)

X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)

print("Balanced Class Distribution")
print(np.unique(y_train_res, return_counts=True))

xgb_smote = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

xgb_smote.fit(
    X_train_res,
    y_train_res
)

y_pred = xgb_smote.predict(X_test)

print(classification_report(y_test, y_pred))

Balanced Class Distribution
(array([0, 1]), array([246, 246]))
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        64
           1       0.96      0.96      0.96       107

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171



## Prepare for ML Flow

In [10]:
models = [

    (
        "Logistic Regression",
        LogisticRegression(
            C=1,
            solver="liblinear",
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost",
        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost + SMOTETomek",
        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )

]

In [11]:
reports = []

for model_name, model, train_set, test_set in models:

    X_tr, y_tr = train_set
    X_te, y_te = test_set

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_te)

    report = classification_report(
        y_te,
        predictions,
        output_dict=True
    )

    reports.append(report)

In [12]:
reports

[{'0': {'precision': 0.9655172413793104,
   'recall': 0.875,
   'f1-score': 0.9180327868852459,
   'support': 64.0},
  '1': {'precision': 0.9292035398230089,
   'recall': 0.9813084112149533,
   'f1-score': 0.9545454545454546,
   'support': 107.0},
  'accuracy': 0.9415204678362573,
  'macro avg': {'precision': 0.9473603906011596,
   'recall': 0.9281542056074766,
   'f1-score': 0.9362891207153503,
   'support': 171.0},
  'weighted avg': {'precision': 0.942794632803145,
   'recall': 0.9415204678362573,
   'f1-score': 0.9408798947194116,
   'support': 171.0}},
 {'0': {'precision': 0.9354838709677419,
   'recall': 0.90625,
   'f1-score': 0.9206349206349206,
   'support': 64.0},
  '1': {'precision': 0.944954128440367,
   'recall': 0.9626168224299065,
   'f1-score': 0.9537037037037037,
   'support': 107.0},
  'accuracy': 0.9415204678362573,
  'macro avg': {'precision': 0.9402189997040544,
   'recall': 0.9344334112149533,
   'f1-score': 0.9371693121693121,
   'support': 171.0},
  'weighted avg

## Exp Tracking ML Flow Local

In [13]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Breast Cancer Classification PBLM 1")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1785610150063, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785610150063, lifecycle_stage='active', name='Breast Cancer Classification PBLM 1', tags={}, trace_location=None, workspace='default'>

In [26]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        #Params
        mlflow.log_param("Model", model_name)

        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())

        #Mts
        mlflow.log_metric(
            "Accuracy",
            report["accuracy"]
        )

        mlflow.log_metric(
            "Precision_Malignant",
            report["0"]["precision"]
        )

        mlflow.log_metric(
            "Recall_Malignant",
            report["0"]["recall"]
        )

        mlflow.log_metric(
            "F1_Malignant",
            report["0"]["f1-score"]
        )

        mlflow.log_metric(
            "Precision_Benign",
            report["1"]["precision"]
        )

        mlflow.log_metric(
            "Recall_Benign",
            report["1"]["recall"]
        )

        mlflow.log_metric(
            "F1_Benign",
            report["1"]["f1-score"]
        )

        mlflow.log_metric(
            "Macro_F1",
            report["macro avg"]["f1-score"]
        )

        mlflow.log_metric(
            "Weighted_F1",
            report["weighted avg"]["f1-score"]
        )

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(
                model,
                artifact_path="model"
            )
        else:
            mlflow.sklearn.log_model(
                model,
                artifact_path="model"
            )

2026/08/02 00:19:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/714ae4f690094f9289edc135c6a2dd1a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/08/02 00:19:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/ee5b32df68bf4975a3ca67906f180125
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/08/02 00:19:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/40e4df10efcf4b2f9e6d71adf4a932b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/08/02 00:19:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost + SMOTETomek at: http://127.0.0.1:5000/#/experiments/2/runs/36eb0478f35b46ba899224cac601c126
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


## Reg teh Best Model

In [15]:
best_index = np.argmax(
    [report["macro avg"]["f1-score"] for report in reports]
)


best_model_name, best_model, _, _ = models[best_index]

best_report = reports[best_index]

print("Model:", best_model_name)
print("Accuracy:", best_report["accuracy"])
print("Macro F1:", best_report["macro avg"]["f1-score"])

Model: XGBoost
Accuracy: 0.9649122807017544
Macro F1: 0.962044983722995


In [16]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    #Params
    mlflow.log_param(
        "Model",
        best_model_name
    )
    mlflow.log_param(
        "Selection_Metric",
        "Macro F1 Score"
    )
    mlflow.log_param(
        "Dataset",
        "Breast Cancer Wisconsin Dataset"
    )
    mlflow.log_params(
        best_model.get_params()
    ) 
    #Metrics
    mlflow.log_metric(
        "Accuracy",
        best_report["accuracy"]
    )

    mlflow.log_metric(
        "Macro_F1",
        best_report["macro avg"]["f1-score"]
    )

    mlflow.log_metric(
        "Weighted_F1",
        best_report["weighted avg"]["f1-score"]
    )


    mlflow.log_metric(
        "Recall_Malignant",
        best_report["0"]["recall"]
    )

    mlflow.log_metric(
        "Precision_Malignant",
        best_report["0"]["precision"]
    )

    mlflow.log_metric(
        "F1_Malignant",
        best_report["0"]["f1-score"]
    )


    mlflow.log_metric(
        "Recall_Benign",
        best_report["1"]["recall"]
    )

    mlflow.log_metric(
        "F1_Benign",
        best_report["1"]["f1-score"]
    )

    mlflow.set_tag(
        "Model_Type",
        best_model_name
    )

    mlflow.set_tag(
        "Stage",
        "Candidate"
    )

    mlflow.set_tag(
        "Task",
        "Binary Classification"
    )

    if "XGBoost" in best_model_name:

        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Breast_Cancer_Best_Model"
        )

    else:

        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Breast_Cancer_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID       :", run_id)
print("Model URI    :", model_uri)
print("Model Name   :", "Breast_Cancer_Best_Model")

2026/08/02 00:42:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'Breast_Cancer_Best_Model' already exists. Creating a new version of this model...
2026/08/02 00:43:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Breast_Cancer_Best_Model, version 2
Created version '2' of model 'Breast_Cancer_Best_Model'.


🏃 View run Champion_XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/a3cf5c440cb04449a1e56e882a0f21a2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Run ID       : a3cf5c440cb04449a1e56e882a0f21a2
Model URI    : models:/m-5d0d05005ccc4c9ebc6d43479cf28de6
Model Name   : Breast_Cancer_Best_Model


## Loading and Pushing to Prod

In [17]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

latest_version = client.get_latest_versions(
    "Breast_Cancer_Best_Model"
)[0].version

print(latest_version)

2


In [18]:
import mlflow
from mlflow.tracking import MlflowClient


mlflow.set_tracking_uri(
    "http://127.0.0.1:5000"
)


client = MlflowClient()

In [19]:
model_name = "Breast_Cancer_Best_Model"


latest_version = client.get_latest_versions(
    model_name
)[0]


print("Model Name:", latest_version.name)
print("Version:", latest_version.version)
print("Stage:", latest_version.current_stage)

Model Name: Breast_Cancer_Best_Model
Version: 2
Stage: None


In [20]:
model_uri = f"models:/{model_name}/{latest_version.version}"


model = mlflow.pyfunc.load_model(
    model_uri
)


print("Model loaded successfully")

Model loaded successfully


In [21]:
predictions = model.predict(
    X_test
)


print(predictions[:10])

[0 1 1 0 0 0 1 0 1 0]


In [22]:
from sklearn.metrics import classification_report
print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

           0       0.98      0.92      0.95        64
           1       0.95      0.99      0.97       107

    accuracy                           0.96       171
   macro avg       0.97      0.96      0.96       171
weighted avg       0.97      0.96      0.96       171



In [23]:
client.update_model_version(
    name=model_name,
    version=latest_version.version,
    description="""
    Champion model for Breast Cancer Classification.

    Tested successfully before production deployment.

    Dataset:
    sklearn Breast Cancer Wisconsin

    Metric:
    Macro F1 Score
    """
)

<ModelVersion: aliases=[], creation_timestamp=1785611587152, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('\n'
 '    Champion model for Breast Cancer Classification.\n'
 '\n'
 '    Tested successfully before production deployment.\n'
 '\n'
 '    Dataset:\n'
 '    sklearn Breast Cancer Wisconsin\n'
 '\n'
 '    Metric:\n'
 '    Macro F1 Score\n'
 '    '), last_updated_timestamp=1785611812869, metrics=None, model_id=None, name='Breast_Cancer_Best_Model', params=None, run_id='a3cf5c440cb04449a1e56e882a0f21a2', run_link='', source='models:/m-5d0d05005ccc4c9ebc6d43479cf28de6', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

In [24]:
client.transition_model_version_stage(
    name=model_name,
    version=latest_version.version,
    stage="Production"
)

Model moved to Production


In [25]:
production_model = mlflow.pyfunc.load_model(
    "models:/Breast_Cancer_Best_Model/Production"
)
prediction = production_model.predict(
    X_test
)
print(prediction[:10])

[0 1 1 0 0 0 1 0 1 0]


In [26]:
production_versions = client.get_latest_versions(
    "Breast_Cancer_Best_Model",
    stages=["Production"]
)
for model in production_versions:
    print(
        "Version:",
        model.version
    )
    print(
        "Run ID:",
        model.run_id
    )
    print(
        "Stage:",
        model.current_stage
    )

Version: 2
Run ID: a3cf5c440cb04449a1e56e882a0f21a2
Stage: Production
